# TL;DR
Read, display, and write PNG, JPEG, and DICOM chest radiographs.

**Inputs:** Practice_PNGandJPG.zip and Practice_DICOM.zip.

**Outputs:** Image copies, DICOM copy, normalized PNG preview, and comparison figures.

Original Colab workflow; saved outputs cleared for publication. See the Python version for local execution.

# Warm-up Exercise 1: Read and Write Chest X-rays

This notebook practices reading, displaying, and writing chest X-rays in PNG, JPG, and DICOM formats. It uses the two ZIP files uploaded to Google Drive.

If your Drive folder has a different name or location, change `DRIVE_FOLDER` in the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_FOLDER = Path('/content/drive/MyDrive/WarmUpData')
PNG_JPG_ZIP = DRIVE_FOLDER / 'Practice_PNGandJPG.zip'
DICOM_ZIP = DRIVE_FOLDER / 'Practice_DICOM.zip'

WORK_DIR = Path('/content/warmup_exercise_1')
PNG_JPG_DIR = WORK_DIR / 'practice_png_jpg'
DICOM_DIR = WORK_DIR / 'practice_dicom'
OUTPUT_DIR = DRIVE_FOLDER / 'exercise_1_output'
OUTPUT_DIR.mkdir(exist_ok=True)

assert PNG_JPG_ZIP.exists(), f'Missing: {PNG_JPG_ZIP}'
assert DICOM_ZIP.exists(), f'Missing: {DICOM_ZIP}'
print('ZIP files found.')

In [ ]:
!pip -q install pydicom

import shutil

if PNG_JPG_DIR.exists():
    shutil.rmtree(PNG_JPG_DIR)
if DICOM_DIR.exists():
    shutil.rmtree(DICOM_DIR)

shutil.unpack_archive(PNG_JPG_ZIP, PNG_JPG_DIR)
shutil.unpack_archive(DICOM_ZIP, DICOM_DIR)
print(f'Extracted files into {WORK_DIR}')

In [ ]:
from collections import Counter

png_jpg_files = [path for path in PNG_JPG_DIR.rglob('*') if path.is_file()]
dicom_files = [path for path in DICOM_DIR.rglob('*') if path.is_file()]

print('PNG/JPG files:', len(png_jpg_files), Counter(path.suffix.lower() for path in png_jpg_files))
print('DICOM files:', len(dicom_files), Counter(path.suffix.lower() or '[no extension]' for path in dicom_files))
print('Example PNG/JPG file:', png_jpg_files[0])
print('Example DICOM file:', dicom_files[0])

## PNG and JPG

The following cell reads one PNG and one JPG image, displays their dimensions and grayscale pixel range, then writes copies to the persistent Drive output folder.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

png_file = next(path for path in png_jpg_files if path.suffix.lower() == '.png')
jpg_file = next(path for path in png_jpg_files if path.suffix.lower() in {'.jpg', '.jpeg'})

png_image = Image.open(png_file).convert('L')
jpg_image = Image.open(jpg_file).convert('L')

for name, image in [('PNG', png_image), ('JPG', jpg_image)]:
    pixels = np.asarray(image)
    print(f'{name}: size={image.size}, mode={image.mode}, pixel range={pixels.min()} to {pixels.max()}')

figure, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(png_image, cmap='gray')
axes[0].set_title('PNG chest X-ray')
axes[1].imshow(jpg_image, cmap='gray')
axes[1].set_title('JPG chest X-ray')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'png_jpg_display.png', dpi=150)
plt.show()

png_image.save(OUTPUT_DIR / 'written_from_png.png')
jpg_image.save(OUTPUT_DIR / 'written_from_jpg.jpg', quality=95)
print('Wrote PNG and JPG copies to:', OUTPUT_DIR)

## DICOM

DICOM stores the pixel data together with medical-image metadata. This cell reads one DICOM image, displays selected metadata and the pixel array, then writes both a DICOM copy and a PNG preview.

In [ ]:
import pydicom

dicom_file = dicom_files[0]
dataset = pydicom.dcmread(dicom_file)
pixels = dataset.pixel_array

print('DICOM file:', dicom_file.name)
print('Modality:', getattr(dataset, 'Modality', 'not recorded'))
print('Rows x Columns:', dataset.Rows, 'x', dataset.Columns)
print('Bits stored:', getattr(dataset, 'BitsStored', 'not recorded'))
print('Pixel range:', pixels.min(), 'to', pixels.max())

plt.figure(figsize=(6, 6))
plt.imshow(pixels, cmap='gray')
plt.title('DICOM chest X-ray')
plt.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dicom_display.png', dpi=150)
plt.show()

# Write a DICOM copy and an 8-bit PNG display copy.
dataset.save_as(OUTPUT_DIR / 'written_dicom_copy.dcm')
scaled = ((pixels - pixels.min()) / (pixels.max() - pixels.min()) * 255).astype(np.uint8)
Image.fromarray(scaled).save(OUTPUT_DIR / 'dicom_as_png.png')
print('Wrote DICOM and PNG output files to:', OUTPUT_DIR)

## Report notes

For the LaTeX report, include the two saved display figures, describe the different file formats, record the image dimensions and pixel ranges printed by the notebook, and explain that DICOM includes metadata in addition to the image pixels.